# Agent Architectures, Memory, and Tools

This notebook uses plain Python, LangChain, and Google Gemini through `langchain-google-genai` to teach agent architectures with real tool calling. The key pattern is: inspect available data, let the model choose and call tools, execute the tool against local data, and ask the model to interpret the actual result.


In [73]:
# Run once per environment.
%pip install -qU pandas pypdf python-dotenv langchain langchain-core langchain-google-genai


Note: you may need to restart the kernel to use updated packages.


An **agent** is an LLM running inside a loop of plain text responses (reasoning) and structured output (acting) which continuously append to the context prompt for the next iteration of the loop. 

An LLM capable of **tool usage** is one that can produce structured output (eg. JSON) that can be intercepted in the loop and be exectuted at runtime.

## Native Tool Calling Versus Tool-Like Text

For this notebook we use Google Gemini through `langchain-google-genai` because the agent examples need native tool calling.

Native tool calling means the model does not merely print something like:

```text
Database_Schema("deseq2_results")
```

Instead, it returns a structured tool-call object. LangChain sees that object, executes the matching Python function, and sends the tool result back to the model. That is the behavior students need to see when learning how agents use tools.

A plain chat endpoint can still be useful for ordinary text generation, but if it only writes tool-looking text, the tool is not executed. For the examples below, Gemini is used for both plain calls and the real agent so the notebook has a single model provider.


## Set Up Gemini And Load All Local Data

This notebook uses `langchain-google-genai` because the agent lesson requires native tool calling. With native tool calling, the model returns a structured request to call `Database_Schema`, `Sample_Column_Values`, `SQL_Query`, or `Retrieve_Context`; LangChain then executes the Python tool and sends the result back to the model.

The `GOOGLE_API_KEY` is loaded from the project `.env` file.


In [74]:
import os
import re
import sqlite3
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

GEMINI_MODEL = "gemini-2.5-flash"

if not os.getenv("GOOGLE_API_KEY"):
    print("Set GOOGLE_API_KEY in the .env file before running Gemini examples.")

model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0,
)

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

DOCS_DIR = Path("../docs")
if not DOCS_DIR.exists():
    DOCS_DIR = Path("docs")

DB_PATH = Path("../agent_teaching.sqlite")
if not Path("../data").exists():
    DB_PATH = Path("agent_teaching.sqlite")
if DB_PATH.exists():
    DB_PATH.unlink()

all_data_files = []
skipped_r_files = []

for path in sorted(DATA_DIR.rglob("*")):
    if path.is_file():
        if path.suffix.lower() == ".r":
            skipped_r_files.append(path)
        else:
            all_data_files.append(path)

print("Files used as local information:")
for path in all_data_files:
    print("-", path)

print("\nR scripts skipped:")
for path in skipped_r_files:
    print("-", path)

print("\nSQLite database file:", DB_PATH)


Files used as local information:
- ../data/.DS_Store
- ../data/airway_counts.csv
- ../data/airway_metadata.csv
- ../data/dea/DESeq2_results_airway_trt_vs_untrt.csv
- ../data/functional/.Rhistory
- ../data/functional/gProfiler_ORA_results.txt
- ../data/functional/topGO_results_BP_all_DE.txt
- ../data/functional/topGO_results_BP_downregulated.txt
- ../data/functional/topGO_results_BP_upregulated.txt

R scripts skipped:
- ../data/dea/DESeq2.R
- ../data/functional/GProfiler.R
- ../data/functional/TopGO_Split.R

SQLite database file: ../agent_teaching.sqlite


In [75]:
tables = {}
text_documents = []
manifest_rows = []

for path in all_data_files:
    relative_path = path.relative_to(DATA_DIR)
    table_name = "__".join(relative_path.with_suffix("").parts)
    table_name = re.sub(r"[^0-9a-zA-Z]+", "_", table_name).strip("_").lower()

    file_info = {
        "file": str(path),
        "table_name": table_name,
        "rows": None,
        "columns": None,
        "loaded_as": None,
        "notes": "",
    }

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        new_columns = []
        for column in df.columns:
            clean_column = re.sub(r"[^0-9a-zA-Z]+", "_", str(column)).strip("_").lower()
            if clean_column == "":
                clean_column = "unnamed_0"
            new_columns.append(clean_column)
        df.columns = new_columns
        if "unnamed_0" in df.columns:
            if path.name == "airway_metadata.csv":
                df = df.rename(columns={"unnamed_0": "sample_id"})
            else:
                df = df.drop(columns=["unnamed_0"])
        tables[table_name] = df
        file_info["rows"] = len(df)
        file_info["columns"] = len(df.columns)
        file_info["loaded_as"] = "table"
        file_info["notes"] = "CSV loaded with pandas."
        text_documents.append({
            "source": str(path),
            "text": "Source: " + str(path) + "\nTable: " + table_name + "\nColumns: " + ", ".join(df.columns) + "\nPreview:\n" + df.head(5).to_string(index=False),
        })

    elif path.suffix.lower() in [".txt", ".tsv"]:
        try:
            df = pd.read_csv(path, sep=None, engine="python")
            new_columns = []
            for column in df.columns:
                clean_column = re.sub(r"[^0-9a-zA-Z]+", "_", str(column)).strip("_").lower()
                if clean_column == "":
                    clean_column = "unnamed_0"
                new_columns.append(clean_column)
            df.columns = new_columns
            tables[table_name] = df
            file_info["rows"] = len(df)
            file_info["columns"] = len(df.columns)
            file_info["loaded_as"] = "table"
            file_info["notes"] = "Text file looked tabular, so it was loaded with pandas."
            text_documents.append({
                "source": str(path),
                "text": "Source: " + str(path) + "\nTable: " + table_name + "\nColumns: " + ", ".join(df.columns) + "\nPreview:\n" + df.head(5).to_string(index=False),
            })
        except Exception as exc:
            text = path.read_text(errors="replace")
            file_info["loaded_as"] = "text"
            file_info["notes"] = "Could not load as table, so it was kept as text: " + repr(exc)
            text_documents.append({"source": str(path), "text": "Source: " + str(path) + "\n" + text[:4000]})

    else:
        try:
            text = path.read_text(errors="replace")
            file_info["loaded_as"] = "text"
            file_info["notes"] = "Read as text."
            text_documents.append({"source": str(path), "text": "Source: " + str(path) + "\n" + text[:4000]})
        except Exception as exc:
            file_info["loaded_as"] = "binary_or_unreadable"
            file_info["notes"] = "Discovered but not readable as text: " + repr(exc)
            text_documents.append({"source": str(path), "text": "Source: " + str(path) + "\nThis file was discovered but could not be read as useful text."})

    manifest_rows.append(file_info)

manifest = pd.DataFrame(manifest_rows)
df = manifest
df.style.set_properties(
    subset=["notes"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,file,table_name,rows,columns,loaded_as,notes
0,../data/.DS_Store,ds_store,nan,nan,text,Read as text.
1,../data/airway_counts.csv,airway_counts,63677.000000,18.000000,table,CSV loaded with pandas.
2,../data/airway_metadata.csv,airway_metadata,8.000000,10.000000,table,CSV loaded with pandas.
3,../data/dea/DESeq2_results_airway_trt_vs_untrt.csv,dea_deseq2_results_airway_trt_vs_untrt,22369.000000,10.000000,table,CSV loaded with pandas.
4,../data/functional/.Rhistory,functional_rhistory,nan,nan,text,Read as text.
5,../data/functional/gProfiler_ORA_results.txt,functional_gprofiler_ora_results,745.000000,13.000000,table,"Text file looked tabular, so it was loaded with pandas."
6,../data/functional/topGO_results_BP_all_DE.txt,functional_topgo_results_bp_all_de,7450.000000,6.000000,table,"Text file looked tabular, so it was loaded with pandas."
7,../data/functional/topGO_results_BP_downregulated.txt,functional_topgo_results_bp_downregulated,7450.000000,6.000000,table,"Text file looked tabular, so it was loaded with pandas."
8,../data/functional/topGO_results_BP_upregulated.txt,functional_topgo_results_bp_upregulated,7450.000000,6.000000,table,"Text file looked tabular, so it was loaded with pandas."


In [76]:
# Make friendly aliases for the main biological tables.
sample_metadata = tables["airway_metadata"]
sample_metadata = sample_metadata.rename(columns={"samplename": "sample_name", "avglength": "avg_length", "sample": "sra_sample"})

gene_counts = tables["airway_counts"]
deseq2_results = tables["dea_deseq2_results_airway_trt_vs_untrt"]
gprofiler_results = tables["functional_gprofiler_ora_results"]
topgo_all_de = tables["functional_topgo_results_bp_all_de"]
topgo_downregulated = tables["functional_topgo_results_bp_downregulated"]
topgo_upregulated = tables["functional_topgo_results_bp_upregulated"]

tables["sample_metadata"] = sample_metadata
tables["gene_counts"] = gene_counts
tables["deseq2_results"] = deseq2_results
tables["gprofiler_results"] = gprofiler_results
tables["topgo_all_de"] = topgo_all_de
tables["topgo_downregulated"] = topgo_downregulated
tables["topgo_upregulated"] = topgo_upregulated

connection = sqlite3.connect(str(DB_PATH))

for table_name in tables:
    tables[table_name].to_sql(table_name, connection, index=False, if_exists="replace")

connection.commit()

print("Tables available for SQL:")
for table_name in sorted(tables):
    print("-", table_name, tables[table_name].shape)

# Build a schema description for the LLM.
# This is the equivalent of a Database_Schema tool.
table_names = pd.read_sql_query(
    "select name from sqlite_master where type = 'table' order by name",
    connection,
)

schema_text = ""
for table_name in table_names["name"]:
    columns = pd.read_sql_query("pragma table_info(" + table_name + ")", connection)
    schema_text = schema_text + "Table: " + table_name + "\n"
    for _, column in columns.iterrows():
        schema_text = schema_text + "- " + column["name"] + " (" + column["type"] + ")\n"
    schema_text = schema_text + "\n"

print("Schema text is ready for natural-language-to-SQL prompts.")

connection.close()
del connection


Tables available for SQL:
- airway_counts (63677, 18)
- airway_metadata (8, 10)
- dea_deseq2_results_airway_trt_vs_untrt (22369, 10)
- deseq2_results (22369, 10)
- functional_gprofiler_ora_results (745, 13)
- functional_topgo_results_bp_all_de (7450, 6)
- functional_topgo_results_bp_downregulated (7450, 6)
- functional_topgo_results_bp_upregulated (7450, 6)
- gene_counts (63677, 18)
- gprofiler_results (745, 13)
- sample_metadata (8, 10)
- topgo_all_de (7450, 6)
- topgo_downregulated (7450, 6)
- topgo_upregulated (7450, 6)
Schema text is ready for natural-language-to-SQL prompts.


In [77]:
# Add the paper PDF to the same simple text collection.
try:
    from pypdf import PdfReader
    pdf_path = DOCS_DIR / "paper.pdf"
    reader = PdfReader(str(pdf_path))
    for page_number, page in enumerate(reader.pages):
        page_text = page.extract_text()
        if page_text:
            text_documents.append({
                "source": str(pdf_path) + " page " + str(page_number + 1),
                "text": page_text[:4000],
            })
    print("Loaded paper pages:", len(reader.pages))
except Exception as exc:
    print("The paper PDF was not loaded:", repr(exc))

print("Text chunks available for retrieval:", len(text_documents))


Loaded paper pages: 13
Text chunks available for retrieval: 22


## 1. Plain LLM

A plain LLM call gets only the question. It does not see the local files, so it may guess.


In [78]:
# question = "Which local files would an agent need to answer: sample composition, top dexamethasone-induced genes, and enriched biological processes?"

# response = model.invoke(question)
# answer = response.content

# df = pd.DataFrame([{"architecture": "plain_llm", "question": question, "answer": answer}])
# df.style.set_properties(
#     subset=["answer"],
#     **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
# )


The LLMs internal knowledge knows what it needs to answer the question, but it doesnt have the *tools* to execute it. Therefore, we can harness its knowledge to let it reason on its own, and only provide it the data and tools for runtime exectution.

## Create Real Agent Tools

This is the key agent pattern used in the omics-query solution.

The user asks one natural-language question. The agent must decide which tools to call:

1. `Database_Schema` to inspect tables and columns.
2. `Sample_Column_Values` to inspect real categorical/text values before filtering.
3. `SQL_Query` to execute a generated read-only SQL query.
4. `Retrieve_Context` when it needs paper or file context.

The Python code below does not hard-code `padj < 0.05`, `padj < 0.01`, or `CRISPLD2` answers. Those are inside the user's natural-language question; the agent has to turn them into tool calls.

In a notebook, the agent tools open a fresh read-only SQLite connection inside each tool call. That avoids the SQLite thread error that happens when a connection created in one notebook thread is reused by a tool running in another thread.


In [79]:
@tool
def Database_Schema(input_text: str = "") -> str:
    """Return all available SQLite tables and their columns. Use this before writing SQL."""
    with sqlite3.connect("file:" + str(DB_PATH) + "?mode=ro", uri=True) as tool_connection:
        table_names = pd.read_sql_query(
            "select name from sqlite_master where type = 'table' order by name",
            tool_connection,
        )

    output = "Available tables and schemas:\n\n"
    for table_name in table_names["name"]:
        with sqlite3.connect("file:" + str(DB_PATH) + "?mode=ro", uri=True) as tool_connection:
            columns = pd.read_sql_query("pragma table_info(" + table_name + ")", tool_connection)
        output = output + "Table: " + table_name + "\n"
        for _, column in columns.iterrows():
            output = output + "- " + column["name"] + " (" + column["type"] + ")\n"
        output = output + "\n"
    return output


@tool
def Sample_Column_Values(input_text: str = "") -> str:
    """Return example distinct values from text columns, so the agent does not guess categorical values."""
    with sqlite3.connect("file:" + str(DB_PATH) + "?mode=ro", uri=True) as tool_connection:
        table_names = pd.read_sql_query(
            "select name from sqlite_master where type = 'table' order by name",
            tool_connection,
        )

    output = "Example values from text columns:\n\n"
    for table_name in table_names["name"]:
        with sqlite3.connect("file:" + str(DB_PATH) + "?mode=ro", uri=True) as tool_connection:
            columns = pd.read_sql_query("pragma table_info(" + table_name + ")", tool_connection)
        for _, column in columns.iterrows():
            column_name = column["name"]
            column_type = str(column["type"]).lower()
            if "text" in column_type:
                try:
                    query = 'select distinct "' + column_name + '" from "' + table_name + '" where "' + column_name + '" is not null limit 8'
                    with sqlite3.connect("file:" + str(DB_PATH) + "?mode=ro", uri=True) as tool_connection:
                        values = pd.read_sql_query(query, tool_connection)
                    if len(values) > 0:
                        output = output + table_name + "." + column_name + ": " + ", ".join(values[column_name].astype(str).tolist()) + "\n"
                except Exception:
                    pass
    return output


@tool
def SQL_Query(query: str) -> str:
    """Execute one read-only SQLite SELECT query and return the actual rows. Input must be a complete SQL SELECT statement."""
    query_clean = query.strip().rstrip(";")
    if not query_clean.lower().startswith("select"):
        return "Rejected: only SELECT queries are allowed."
    forbidden_words = ["drop", "delete", "insert", "update", "alter", "create", "replace"]
    for word in forbidden_words:
        if word in query_clean.lower().split():
            return "Rejected: this query contains a forbidden SQL keyword."
    try:
        with sqlite3.connect("file:" + str(DB_PATH) + "?mode=ro", uri=True) as tool_connection:
            result = pd.read_sql_query(query_clean, tool_connection)
        if len(result) == 0:
            return "Query returned 0 rows."
        return "Query returned " + str(len(result)) + " rows. Showing up to 30 rows:\n" + result.head(30).to_string(index=False)
    except Exception as exc:
        return "SQL error: " + repr(exc)


@tool
def Retrieve_Context(query: str) -> str:
    """Retrieve relevant local file or paper context using simple keyword matching."""
    query_words = query.lower().split()
    scored_documents = []
    for document in text_documents:
        text = document["text"].lower()
        score = 0
        for word in query_words:
            score = score + text.count(word)
        scored_documents.append({"score": score, "source": document["source"], "text": document["text"]})
    scored_documents = sorted(scored_documents, key=lambda row: row["score"], reverse=True)

    output = ""
    for document in scored_documents[:5]:
        output = output + "\n\nSOURCE: " + document["source"] + "\n" + document["text"][:1500]
    return output


agent_tools = [Database_Schema, Sample_Column_Values, SQL_Query, Retrieve_Context]

print("Tools available to the agent:")
for agent_tool in agent_tools:
    print("-", agent_tool.name)


Tools available to the agent:
- Database_Schema
- Sample_Column_Values
- SQL_Query
- Retrieve_Context


## Build The Agent

This is the actual agent. The notebook does not write the SQL query for the question. The agent receives the natural-language question and has to use its tools.

The agent is composed of:
- Model: the LLM which reasons and orchestrates
- Agent tools: the tools the LLM has at its disposal which we have coded above
- System prompt: general guidance on how to act from how to use the data, how to use the tools, and how to format its final response

In [80]:
system_prompt = """
You are an RNA-seq teaching agent.

Answer questions by using tools, not by guessing.

Required workflow for data questions:
1. First call Database_Schema to inspect available tables and columns.
2. Then call Sample_Column_Values before filtering text/categorical columns such as gene symbols, treatment labels, GO terms, or file-derived labels.
3. Then write the correct SQLite SELECT query and call SQL_Query.
4. If biological interpretation or source context is needed, call Retrieve_Context.
5. Give a final answer based only on tool results from this turn.

Important table guidance for this notebook:
- deseq2_results contains differential expression columns including gene_id, symbol, log2foldchange, pvalue, and padj.
- sample_metadata contains sample information including sample_id, cell, and dex.
- gene_counts contains gene annotation columns plus sample-count columns named like srr1039508.
- topgo_upregulated, topgo_downregulated, topgo_all_de, and gprofiler_results contain enrichment results.
- Use padj for adjusted p-value thresholds in deseq2_results.
- Use SELECT only. Never modify data.
- Always include a short biological interpretation and a caveat.
"""

rna_seq_agent = create_agent(
    model,
    agent_tools,
    system_prompt=system_prompt,
)

print("Agent is ready. This version uses Gemini native tool calling through langchain-google-genai.")


Agent is ready. This version uses Gemini native tool calling through langchain-google-genai.


## Ask Natural-Language Questions

Now the user can ask normal questions. The agent should inspect schema, write SQL, run SQL, and answer.


In [81]:
question = "How many genes are significant at padj < 0.05 versus padj < 0.01?"

# With a native tool-calling model, this stream should show actual tool-call messages
# and tool observations, not only text such as ```tool_code ... ```.
for event in rna_seq_agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()


================================ Human Message =================================

How many genes are significant at padj < 0.05 versus padj < 0.01?


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 15.565978461s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

## More Natural-Language Questions

These examples use the same tools. The notebook does not change Python code to handle each question.


In [72]:
questions = [
    "What is the DESeq2 result for CRISPLD2?",
    "Which 10 genes have the largest positive log2FoldChange among genes with padj < 0.01?",
    "Which biological processes are enriched among upregulated genes?",
]

from IPython.display import Markdown, display

for question in questions:
    result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": question}]})
    content = result["messages"][-1].content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + "\n\n"
    else:
        answer_text = str(content)

    display(Markdown("### Question\n\n" + question))
    display(Markdown("### Agent answer\n\n" + answer_text.strip()))
    display(Markdown("---"))


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 48.984368636s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

## Architecture Comparison

The remaining cells compare architectural ideas using the same real tools. The important point is that the agent receives natural language and decides how to use `Database_Schema`, `Sample_Column_Values`, `SQL_Query`, and `Retrieve_Context`.


In [70]:
question = "Compare significant genes at padj < 0.05 and padj < 0.01, then explain what this means biologically."

result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": question}]})

df = pd.DataFrame([{
    "architecture": "tool_using_agent",
    "question": question,
    "top_context": result["messages"][-1].content,
}])
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


KeyboardInterrupt: 

## Router Agent

A router can choose whether a question should go to SQL tools, retrieval, or both. Here we still use the same real agent for the final work.


In [ ]:
questions = [
    "Which local tables are available?",
    "How many genes are significant at padj < 0.05 versus padj < 0.01?",
    "What is the DESeq2 result for CRISPLD2?",
    "What does the paper say about glucocorticoid response in airway smooth muscle?",
]

from IPython.display import Markdown, display

for question in questions:
    lower_question = question.lower()
    if "paper" in lower_question or "say about" in lower_question:
        expected_route = "retrieval_plus_answer"
    elif "tables" in lower_question or "available" in lower_question:
        expected_route = "schema"
    else:
        expected_route = "schema_plus_sql"

    result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": question}]})
    final_message = result["messages"][-1]
    content = final_message.content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + "\n\n"
    else:
        answer_text = str(content)

    display(Markdown("### Question\n\n" + question))
    display(Markdown("**Expected route:** " + expected_route))
    display(Markdown("### Agent answer\n\n" + answer_text.strip()))
    display(Markdown("---"))


## Sequential Chain

A sequential chain is more deterministic: ask the agent focused subquestions, then ask it to combine the answers.


In [ ]:
subquestions = [
    "How many samples are in each dex treatment group?",
    "How many genes are significant at padj < 0.05 versus padj < 0.01?",
    "Which 8 genes have the largest positive log2FoldChange among genes with padj < 0.01?",
    "Which topGO biological processes are enriched among upregulated genes?",
]

from IPython.display import Markdown, display

subanswers = ""
for subquestion in subquestions:
    result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": subquestion}]})
    content = result["messages"][-1].content
    answer_text = ""
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                answer_text = answer_text + block.get("text", "") + "\n\n"
    else:
        answer_text = str(content)
    subanswers = subanswers + "\n\nQuestion: " + subquestion + "\nAnswer:\n" + answer_text.strip()

final_question = "Summarize these subanswers as a compact teaching answer. Separate exact SQL-derived facts from biological interpretation."
result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": final_question + subanswers}]})

content = result["messages"][-1].content
answer_text = ""
if isinstance(content, list):
    for block in content:
        if isinstance(block, dict) and block.get("type") == "text":
            answer_text = answer_text + block.get("text", "") + "\n\n"
else:
    answer_text = str(content)

display(Markdown("### Final answer\n\n" + answer_text.strip()))


## 9. Memory Techniques


### i. No Memory

Each model call is independent.


In [ ]:
first_response = model.invoke("The local dataset is the airway RNA-seq dataset comparing dexamethasone-treated and untreated human airway smooth muscle cell samples.")
followup_response = model.invoke("What treatment comparison is the local dataset about?")

df = pd.DataFrame([
    {"turn": "first", "answer": first_response.content},
    {"turn": "followup_without_memory", "answer": followup_response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### ii. Conversation Memory

Now the follow-up question is sent together with earlier messages.


In [ ]:
messages = [
    {"role": "user", "content": "The local dataset compares dexamethasone-treated and untreated airway smooth muscle cell samples."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "What treatment comparison is the local dataset about?"},
]

response = model.invoke(messages)

df = pd.DataFrame([{"memory": "conversation", "question": messages[-1]["content"], "answer": response.content}])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### iii. Windowed Memory

Windowed memory keeps only recent messages. It can lose useful earlier details.


In [ ]:
full_history = [
    {"role": "user", "content": "The local files include airway_counts.csv, airway_metadata.csv, data/dea/DESeq2_results_airway_trt_vs_untrt.csv, and functional enrichment TXT files from gProfiler and topGO."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "The key treatment variable is the dex column with trt and untrt groups, and enrichment files help interpret gene sets rather than measure expression directly."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "Which local files define the treatment comparison, measured gene expression, differential expression, and functional interpretation?"},
]

short_window_response = model.invoke(full_history[-3:])
full_conversation_response = model.invoke(full_history)

df = pd.DataFrame([
    {"memory": "short_window", "answer": short_window_response.content},
    {"memory": "full_conversation", "answer": full_conversation_response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 10. Tool Usage


### i. No Tools

The model gets no local table values.


In [ ]:
question = "What is the DESeq2 log2FoldChange and adjusted p-value for CRISPLD2, and which upregulated biological processes are enriched in the local airway dataset?"

response = model.invoke(question)

df = pd.DataFrame([{"tool_access": "no_tools", "question": question, "top_context": response.content}])
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### ii. Python Tool

Plain pandas code can calculate exact summaries.


In [ ]:
question = "What is the DESeq2 result for CRISPLD2?"

result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": question}]})
content = result["messages"][-1].content
answer_text = ""
if isinstance(content, list):
    for block in content:
        if isinstance(block, dict) and block.get("type") == "text":
            answer_text = answer_text + block.get("text", "") + "\n\n"
else:
    answer_text = str(content)

from IPython.display import Markdown, display
display(Markdown("### Question\n\n" + question))
display(Markdown("### Agent answer\n\n" + answer_text.strip()))


### iii. SQL Tool

SQL is useful for exact structured questions.


In [ ]:
question = "How many genes are significant at padj < 0.01?"

result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": question}]})
content = result["messages"][-1].content
answer_text = ""
if isinstance(content, list):
    for block in content:
        if isinstance(block, dict) and block.get("type") == "text":
            answer_text = answer_text + block.get("text", "") + "\n\n"
else:
    answer_text = str(content)

from IPython.display import Markdown, display
display(Markdown("### Question\n\n" + question))
display(Markdown("### Agent answer\n\n" + answer_text.strip()))


### iv. Multiple Tools

The strongest answers combine several local evidence sources.


In [ ]:
question = "Combine CRISPLD2 differential expression, significant gene counts at padj < 0.05 versus 0.01, and upregulated topGO enrichment terms."

result = rna_seq_agent.invoke({"messages": [{"role": "user", "content": question}]})
content = result["messages"][-1].content
answer_text = ""
if isinstance(content, list):
    for block in content:
        if isinstance(block, dict) and block.get("type") == "text":
            answer_text = answer_text + block.get("text", "") + "\n\n"
else:
    answer_text = str(content)

from IPython.display import Markdown, display
display(Markdown("### Question\n\n" + question))
display(Markdown("### Agent answer\n\n" + answer_text.strip()))


### v. Tool Descriptions

Clear descriptions help an agent choose the right evidence source. Here we show the same idea with simple labels.


In [ ]:
tool_descriptions = pd.DataFrame([
    {
        "tool_name": "Database_Schema",
        "clear_description": "Use first for data questions. It tells the agent which tables and columns exist before SQL is written.",
        "poor_description": "Show data info.",
    },
    {
        "tool_name": "Sample_Column_Values",
        "clear_description": "Use before WHERE filters on text columns. It prevents guessing values such as gene symbols, treatment labels, or GO terms.",
        "poor_description": "Show examples.",
    },
    {
        "tool_name": "SQL_Query",
        "clear_description": "Run the agent-generated read-only SELECT query and return actual rows. This is where exact numerical answers come from.",
        "poor_description": "Search gene stuff.",
    },
    {
        "tool_name": "Retrieve_Context",
        "clear_description": "Retrieve paper or file context when the question asks for interpretation or provenance beyond exact table values.",
        "poor_description": "Search text.",
    },
])

df = tool_descriptions
df.style.set_properties(
    subset=["clear_description", "poor_description"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)
